# RETINA-AI · Kişi 4: Derin Öğrenme (EfficientNet-B0 + Grad-CAM)

**Çalıştırmadan önce:** `Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU`

**Bölme:** Varsayılan olarak `veri_bolme_v2.csv` kullanılır (sadece 512×512, sınıf başı 288, hasta bazlı).
Eski bölme (`veri_bolme.csv`) 256×256 augmentasyon kopyaları yüzünden sızıntı ve çözünürlük kısayolu içeriyordu; sadece karşılaştırma için durur.

Akış:
1. Drive + kod, bölme seçimi
2. Veriyi yerel diske kopyalama
3. Hızlı deneme
4. Tam eğitim (model zaten varsa atlanabilir)
5. Test tahminleri + Grad-CAM · 5b kenar testi · 5c çözünürlük testi · 5d 5-katlı çapraz doğrulama
6. Çıktıları Drive'a kaydetme

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys
from pathlib import Path

# Kodu GitHub'dan cek: DAL varsa onu, yoksa main'i kullan
DAL = 'feature/derin-ogrenme-cnn'   # merge edildikten sonra 'main' yapabilirsiniz
!test -d /content/retina-ai || git clone -q https://github.com/blosny/retina-ai-ocular-disease-classification.git /content/retina-ai
%cd /content/retina-ai
!git fetch -q origin {DAL} && git checkout -q -B {DAL} FETCH_HEAD || (git checkout -q main && git pull -q origin main)
!git log --oneline -1

PROJE = Path('/content/retina-ai')
DRIVE = Path('/content/drive/MyDrive/retina-ai-ocular-disease-classification')
sys.path.insert(0, str(PROJE))

if not (PROJE / 'src' / 'cnn_models.py').exists():
    raise FileNotFoundError(f"src/cnn_models.py bulunamadi: '{DAL}' dali GitHub'a push edilmis mi kontrol edin.")
if not DRIVE.exists():
    raise FileNotFoundError("Drive klasoru bulunamadi: paylasilan klasoru 'Drive'a kisayol ekle -> Drive'im' ile ekleyin.")
print('Kod ve Drive hazir.')

# ---- Bolme secimi ----
from src import cnn_models as cnn
cnn.BOLME_DOSYASI = 'veri_bolme_v2.csv'   # eski bolme icin: 'veri_bolme.csv'
cnn.DOSYA_EKI     = '_v2'                 # rapor dosyalarinin sonuna eklenir
print('Bolme:', cnn.BOLME_DOSYASI)

## 2. Veriyi yerel diske kopyala
Görüntülerin `<sınıf>/<dosya>.jpg` yapısında olduğu klasörü bulur (`data/raw`, `archive/dataset` veya `dataset`). Zip olarak durduysa açar.

In [ ]:
import shutil, zipfile
YEREL = Path('/content/dataset')

if not (YEREL / 'glaucoma').is_dir():
    adaylar = [DRIVE/'data'/'raw', DRIVE/'archive'/'dataset', DRIVE/'dataset', DRIVE/'data'/'raw'/'dataset']
    kaynak = next((a for a in adaylar if (a/'glaucoma').is_dir()), None)
    zipler = list(DRIVE.rglob('*.zip'))
    if kaynak:
        print('Kopyalaniyor:', kaynak); shutil.copytree(kaynak, YEREL)
    elif zipler:
        print('Zip aciliyor:', zipler[0])
        zipfile.ZipFile(zipler[0]).extractall('/content/_zip')
        ic = next(p.parent for p in Path('/content/_zip').rglob('glaucoma') if p.is_dir())
        shutil.move(str(ic), str(YEREL))
    else:
        raise FileNotFoundError('Drive klasorunde goruntuler bulunamadi, klasor yapisini kontrol edin.')

for s in ['normal', 'diabetic_retinopathy', 'glaucoma', 'cataract']:
    print(s, len(list((YEREL/s).glob('*'))))

## 3. Hızlı deneme (2-3 dakika)
Hata çıkmazsa tam eğitime geç.

In [ ]:
cnn.egit(PROJE, veri_koku=YEREL, model_yolu=PROJE/'models'/'deneme.pth',
         epoch=2, ornek_sayisi=200, rapor_yaz=False)

## 4. Tam eğitim
- 384×384 girdi, batch 16, AdamW (lr=3e-4), en fazla 30 epoch, erken durdurma (sabır 5)
- En iyi model ortak Drive'daki `models/` klasörüne kaydedilir.
- **Model zaten varsa** (`models/model_efficientnet_v2.pth`) `EGIT = False` yapıp bu hücreyi çalıştırın; sadece yolu tanımlar.

In [ ]:
KAYIT = DRIVE   # ortak klasore yazma izniniz yoksa: Path('/content/drive/MyDrive/retina-ai-k4')
MODEL_YOLU = KAYIT / 'models' / f'model_efficientnet{cnn.DOSYA_EKI}.pth'
EGIT = not MODEL_YOLU.exists()
print('Model:', MODEL_YOLU, '| egitilecek mi:', EGIT)

if EGIT:
    gecmis = cnn.egit(PROJE, veri_koku=YEREL, model_yolu=MODEL_YOLU,
                      epoch=30, sabir=5, batch=16, boyut=384, girdi_modu='rgb')
    from IPython.display import Image, display
    display(Image(str(PROJE/'reports'/'figures'/f'cnn_egitim_grafigi{cnn.DOSYA_EKI}.png')))

## 5. Test tahminleri + Grad-CAM
- `reports/cnn_test_tahminleri.csv`: her test görüntüsü için gerçek sınıf, tahmin ve 4 olasılık (Kişi 5 metrikleri buradan hesaplar)
- `reports/figures/gradcam_ornekleri.png`: her sınıftan 1 doğru + 1 yanlış tahmin

**Grad-CAM'i yorumlarken:** Isı optik disk (glokom), lezyonlar (DR) veya tüm retina (katarakt) üzerinde mi? Siyah kenarlarda veya köşelerdeyse model kamerayı/kaynağı öğrenmiş olabilir.

In [ ]:
from IPython.display import Image, display
tahminler = cnn.test_tahminlerini_kaydet(PROJE, veri_koku=YEREL, model_yolu=MODEL_YOLU)
print(tahminler.groupby('gercek')['dogru'].mean().round(3))

cnn.gradcam_figuru_uret(tahminler, PROJE, veri_koku=YEREL, model_yolu=MODEL_YOLU)
display(Image(str(PROJE/'reports'/'figures'/f'gradcam_ornekleri{cnn.DOSYA_EKI}.png')))

## 5b. Kenar testi (kaynak/kamera kisayolu kontrolu)
Test goruntulerinin dis halkasi siyahla kapatilip model tekrar calistirilir.
- Kenar kapatilinca bir sinifin dogrulugu digerlerinden cok daha fazla dusuyorsa model o sinifi kenar/kamera izlerinden taniyor olabilir.
- `sadece_kenar` kosulunda retinanin ortasi tamamen siyahtir; bu goruntu "detay yok" oldugu icin katarakta benzer, tek basina kisayol kaniti degildir.

In [ ]:
import numpy as np, pandas as pd, torch, torch.nn.functional as F

model = cnn.model_yukle(MODEL_YOLU)
donusum = cnn.donusum_olustur(model.bilgi['goruntu_boyutu'], egitim=False)
cihaz = next(model.parameters()).device
df = pd.read_csv(PROJE/'data'/'splits'/cnn.BOLME_DOSYASI)
df = df[df['split'] == 'test'].reset_index(drop=True)

yy, xx = np.mgrid[:512, :512]
uzaklik = np.sqrt((xx - 256)**2 + (yy - 256)**2) / 256   # 0 = merkez, 1 = kenar
KOSULLAR = {                                              # True olan pikseller KALIR
    'orijinal':          np.ones((512, 512), bool),
    'kenar_kapali_%15':  uzaklik <= 0.85,
    'kenar_kapali_%30':  uzaklik <= 0.70,
    'sadece_kenar':      uzaklik > 0.85,
}

satirlar = []
with torch.no_grad():
    for _, s in df.iterrows():
        rgb = cnn.cnn_girdisi_hazirla(YEREL / s['relative_path'])
        x = torch.stack([donusum(rgb * m[..., None].astype(np.uint8)) for m in KOSULLAR.values()]).to(cihaz)
        tahminler_k = F.softmax(model(x), 1).argmax(1).cpu().numpy()
        for ad, t in zip(KOSULLAR, tahminler_k):
            satirlar.append({'gercek': s['class_name'], 'kosul': ad, 'tahmin': cnn.SINIFLAR[t]})

sonuc = pd.DataFrame(satirlar)
sonuc['dogru'] = sonuc['gercek'] == sonuc['tahmin']

print('1) SINIF BAZINDA DOGRULUK (%)')
tablo = (sonuc.pivot_table(index='gercek', columns='kosul', values='dogru', aggfunc='mean') * 100).round(1)
display(tablo[list(KOSULLAR)])

print('\n2) SADECE KENAR VERILINCE "KATARAKT" DENEN ORAN (%)')
kenar = sonuc[sonuc['kosul'] == 'sadece_kenar']
display((kenar.assign(katarakt=kenar['tahmin'] == 'cataract')
             .groupby('gercek')['katarakt'].mean() * 100).round(1))

sonuc.to_csv(PROJE/'reports'/f'cnn_kenar_testi{cnn.DOSYA_EKI}.csv', index=False)
tablo.to_csv(PROJE/'reports'/f'cnn_kenar_testi_ozet{cnn.DOSYA_EKI}.csv')

## 5c. Çözünürlük testi (kısayol kontrolü)
Test görüntüleri 256×256'ya düşürülüp (JPEG kalite 75) tekrar 512×512'ye büyütülür. Eski bölmede DR ve normal görüntüleri bu işlemle neredeyse tamamen katarakt/glokoma kaymıştı (çözünürlük kısayolu).
Temiz bir modelde tahminlerin büyük ölçüde aynı kalması beklenir; kataraktta bir miktar kayma doğaldır çünkü küçültme bulanıklık da ekler.

In [ ]:
import numpy as np, pandas as pd, cv2, torch, torch.nn.functional as F

model = cnn.model_yukle(MODEL_YOLU)
donusum = cnn.donusum_olustur(model.bilgi['goruntu_boyutu'], egitim=False)
cihaz = next(model.parameters()).device
test = pd.read_csv(PROJE/'data'/'splits'/cnn.BOLME_DOSYASI).query("split == 'test'").reset_index(drop=True)

def dusuk_cozunurluk(rgb):
    k = cv2.resize(rgb, (256, 256), interpolation=cv2.INTER_AREA)
    _, buf = cv2.imencode('.jpg', cv2.cvtColor(k, cv2.COLOR_RGB2BGR), [cv2.IMWRITE_JPEG_QUALITY, 75])
    k = cv2.cvtColor(cv2.imdecode(buf, 1), cv2.COLOR_BGR2RGB)
    return cv2.resize(k, (512, 512), interpolation=cv2.INTER_AREA)

def sinif(rgb_listesi):
    x = torch.stack([donusum(r) for r in rgb_listesi]).to(cihaz)
    with torch.no_grad():
        return [cnn.SINIFLAR[i] for i in F.softmax(model(x), 1).argmax(1).cpu().numpy()]

orj, dus = [], []
for i in range(0, len(test), 32):
    rgb = [cnn.cnn_girdisi_hazirla(YEREL/p) for p in test.relative_path[i:i+32]]
    orj += sinif(rgb); dus += sinif([dusuk_cozunurluk(r) for r in rgb])
test['tahmin'], test['tahmin_256'] = orj, dus

print('ORIJINAL tahmin dagilimi (%)')
display((pd.crosstab(test.class_name, test.tahmin, normalize='index') * 100).round(1))
print('256x256e DUSURULUNCE tahmin dagilimi (%)')
display((pd.crosstab(test.class_name, test.tahmin_256, normalize='index') * 100).round(1))
print('Dogruluk: orijinal %.1f | 256 %.1f' % ((test.tahmin == test.class_name).mean()*100,
                                              (test.tahmin_256 == test.class_name).mean()*100))
test[['image_name', 'class_name', 'tahmin', 'tahmin_256']].to_csv(
    PROJE/'reports'/f'cnn_cozunurluk_testi{cnn.DOSYA_EKI}.csv', index=False)

## 5d. 5-katlı çapraz doğrulama (~15 dk)
Test seti sınıf başına ~43 görüntü olduğu için tek bölmenin sonucu ±6 puan oynayabilir.
Tüm 1152 görüntü hasta bazlı 5 kata bölünür; her katta 3 kat eğitim, 1 kat doğrulama, 1 kat test olur ve her görüntü tam bir kez test edilir. Sonuç: ortalama ± standart sapma.
Kat modelleri geçici klasöre kaydedilir; arayüzde kullanılan model 4. hücredeki modeldir.

In [ ]:
ozet = cnn.capraz_dogrulama(PROJE, veri_koku=YEREL, k=5,
                            epoch=30, sabir=5, batch=16, boyut=384)
display((ozet * 100).round(1))

## Tek görüntü denemesi (arayüzdeki çağrının aynısı)

In [ ]:
import cv2, matplotlib.pyplot as plt
ornek = sorted((YEREL/'glaucoma').glob('*'))[0]
bgr = cv2.imread(str(ornek))

model = cnn.model_yukle(MODEL_YOLU)
print(cnn.tahmin(bgr, model))
harita = cnn.gradcam(bgr, model=model)
rgb = cnn.cnn_girdisi_hazirla(bgr)
plt.figure(figsize=(8, 4))
plt.subplot(1, 2, 1); plt.imshow(rgb); plt.axis('off')
plt.subplot(1, 2, 2); plt.imshow(cnn.isi_haritasi_bindir(rgb, harita)); plt.axis('off')
plt.show()

## 6. Çıktıları Drive'a kopyala

In [ ]:
import shutil
EK = cnn.DOSYA_EKI
hedef = KAYIT / 'reports'
(hedef / 'figures').mkdir(parents=True, exist_ok=True)
for ad in [f'cnn_egitim_gecmisi{EK}.json', f'cnn_test_tahminleri{EK}.csv',
           f'cnn_kenar_testi{EK}.csv', f'cnn_kenar_testi_ozet{EK}.csv',
           f'cnn_cozunurluk_testi{EK}.csv',
           f'cnn_capraz_dogrulama{EK}.csv', f'cnn_capraz_tahminleri{EK}.csv']:
    kaynak = PROJE/'reports'/ad
    if kaynak.exists():
        shutil.copy(kaynak, hedef/ad); print('kaydedildi:', ad)
for ad in [f'cnn_egitim_grafigi{EK}.png', f'gradcam_ornekleri{EK}.png']:
    kaynak = PROJE/'reports'/'figures'/ad
    if kaynak.exists():
        shutil.copy(kaynak, hedef/'figures'/ad); print('kaydedildi: figures/' + ad)